# 1. A first model

A bank with tellers: customers arrive at random, wait for a free teller, are served and leave.
This notebook builds the model, runs it once, looks inside the run, and replays it.

Install with `pip install "simulsi[viz]"`.

In [ ]:
from simulsi import Model, Parameter, Simulation
from simulsi.randomness import Exponential, LogNormal

A model is a function that sets up a `Simulation` from parameters. Processes are Python generators:
`yield` a number to let time pass, or yield a request to wait for a resource.

In [ ]:
def bank(sim: Simulation, p) -> None:
    tellers = sim.resource("teller", p.tellers)
    arrivals, service = sim.stream("arrivals"), sim.stream("service")
    service_time = LogNormal.from_moments(mean=p.service_mean, std=p.service_mean / 2)

    def customer(sim):
        yield from sim.use(tellers, service_time.sample(service))

    def source(sim):
        while True:
            yield Exponential(rate=p.arrival_rate).sample(arrivals)
            sim.process(customer(sim))

    sim.process(source(sim))


model = Model(
    bank,
    name="bank",
    duration=480,
    warmup=60,
    parameters=[
        Parameter("tellers", 3, "int", low=1),
        Parameter("arrival_rate", 0.5, "float", low=0),
        Parameter("service_mean", 5.0, "float", low=0),
    ],
)

Run it once. In Jupyter the result shows as a table of metrics.

In [ ]:
run = model.simulate(seed=1)
run

In [ ]:
m = run.metrics
print(
    f"utilization {m['resource.teller.utilization']:.1%}, mean wait {m['resource.teller.wait.mean']:.2f} min"
)

## Look inside a run

`trace=True` records an event log; `series` holds queue length and busy tellers over time.

In [ ]:
traced = model.simulate(seed=1, trace=True, duration=120)
print(len(traced.log.records), "events logged")
for rec in traced.log.records[:5]:
    print(rec.timestamp, rec.event_type, rec.resource)

In [ ]:
import matplotlib.pyplot as plt

t, q = zip(*traced.series["resource.teller.queue_length"], strict=True)
plt.step(t, q, where="post")
plt.xlabel("minutes")
plt.ylabel("customers waiting")
plt.show()

## Replay it in the dashboard

`simulsi ui --model this_file.py` (or `builtin:` models) opens the local dashboard, where
*Trace & replay a run* plays the run back: busy and idle tellers, the waiting line and the latest events.